In [20]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler


In [21]:
df = pd.read_csv("../data/processed/cleaned_online_retail.csv")

print("Rows:", len(df))
print("Columns:", df.shape[1])

Rows: 400916
Columns: 13


In [22]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice,Year,Month,Day,Hour
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4,2009,12,1,7
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0,2009,12,1,7
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0,2009,12,1,7
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8,2009,12,1,7
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0,2009,12,1,7


In [23]:
print(df.columns.tolist())

['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country', 'TotalPrice', 'Year', 'Month', 'Day', 'Hour']


In [18]:
df.info()
df.describe()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400916 entries, 0 to 400915
Data columns (total 13 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    400916 non-null  int64  
 1   StockCode    400916 non-null  object 
 2   Description  400916 non-null  object 
 3   Quantity     400916 non-null  int64  
 4   InvoiceDate  400916 non-null  object 
 5   UnitPrice    400916 non-null  float64
 6   CustomerID   400916 non-null  int64  
 7   Country      400916 non-null  object 
 8   TotalPrice   400916 non-null  float64
 9   Year         400916 non-null  int64  
 10  Month        400916 non-null  int64  
 11  Day          400916 non-null  int64  
 12  Hour         400916 non-null  int64  
dtypes: float64(2), int64(7), object(4)
memory usage: 39.8+ MB


,InvoiceNo,Quantity,UnitPrice,CustomerID,TotalPrice,Year,Month,Day,Hour
count,400916.000000,400916.000000,400916.000000,400916.000000,400916.000000,400916.000000,400916.000000,400916.000000,400916.000000
mean,514731.380771,13.767418,3.305826,15361.544074,21.945330,2009.924493,7.398166,15.365368,12.865191
std,14090.603233,97.638385,35.047719,1680.635823,77.758075,0.264208,3.472188,8.735086,2.306906
min,489434.000000,1.000000,0.001000,12346.000000,0.001000,2009.000000,1.000000,1.000000,7.000000
25%,502752.000000,2.000000,1.250000,13985.000000,5.000000,2010.000000,4.000000,8.000000,11.000000
50%,515192.000000,5.000000,1.950000,15311.000000,12.500000,2010.000000,8.000000,15.000000,13.000000
75%,527065.500000,12.000000,3.750000,16805.000000,19.500000,2010.000000,11.000000,23.000000,14.000000
max,538171.000000,19152.000000,10953.500000,18287.000000,15818.400000,2010.000000,12.000000,31.000000,20.000000


In [24]:
customer_df = df.groupby("CustomerID").agg(
    TotalSpend=("TotalPrice", "sum"),
    TotalQuantity=("Quantity", "sum"),
    NumberOfTransactions=("InvoiceNo", "nunique"),
    AverageOrderValue=("TotalPrice", "mean"),
    AverageUnitPrice=("UnitPrice", "mean")
).reset_index()

customer_df.head()

,CustomerID,TotalSpend,TotalQuantity,NumberOfTransactions,AverageOrderValue,AverageUnitPrice
0,12346,372.86,70,11,11.298788,6.253333
1,12347,1323.32,828,2,18.638310,2.295070
2,12348,222.16,373,1,11.108000,0.719500
3,12349,2671.14,993,3,26.187647,8.581765
4,12351,300.93,261,1,14.330000,2.355238


In [26]:
print("Original transaction data:", df.shape)
print("Customer-level data:", customer_df.shape)

Original transaction data: (400916, 13)
Customer-level data: (4312, 6)


In [29]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

latest_date = df["InvoiceDate"].max()
reference_date = latest_date + pd.Timedelta(days=1)

recency = df.groupby("CustomerID")["InvoiceDate"].max().reset_index()
recency["RecencyDays"] = (reference_date - recency["InvoiceDate"]).dt.days
recency = recency[["CustomerID", "RecencyDays"]]

customer_df = customer_df.merge(recency, on="CustomerID", how="left")

customer_df.head()

,CustomerID,TotalSpend,TotalQuantity,NumberOfTransactions,AverageOrderValue,AverageUnitPrice,RecencyDays
0,12346,372.86,70,11,11.298788,6.253333,165
1,12347,1323.32,828,2,18.638310,2.295070,3
2,12348,222.16,373,1,11.108000,0.719500,74
3,12349,2671.14,993,3,26.187647,8.581765,43
4,12351,300.93,261,1,14.330000,2.355238,11


In [31]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

In [33]:
customer_df.shape

(4312, 6)

In [35]:
customer_df.columns.tolist()

['CustomerID',
 'TotalSpend',
 'TotalQuantity',
 'NumberOfTransactions',
 'AverageOrderValue',
 'AverageUnitPrice']

In [37]:
customer_df.isnull().sum()

CustomerID              0
TotalSpend              0
TotalQuantity           0
NumberOfTransactions    0
AverageOrderValue       0
AverageUnitPrice        0
dtype: int64

In [38]:
customer_df = df.groupby("CustomerID").agg(
    TotalSpend=("TotalPrice", "sum"),
    TotalQuantity=("Quantity", "sum"),
    NumberOfTransactions=("InvoiceNo", "nunique"),
    AverageOrderValue=("TotalPrice", "mean"),
    AverageUnitPrice=("UnitPrice", "mean")
).reset_index()

customer_df.head()

,CustomerID,TotalSpend,TotalQuantity,NumberOfTransactions,AverageOrderValue,AverageUnitPrice
0,12346,372.86,70,11,11.298788,6.253333
1,12347,1323.32,828,2,18.638310,2.295070
2,12348,222.16,373,1,11.108000,0.719500
3,12349,2671.14,993,3,26.187647,8.581765
4,12351,300.93,261,1,14.330000,2.355238
